In [2]:
import pandas as pd
import joblib
import numpy as np

# =====================================================
# LOAD MODEL
# =====================================================

model = joblib.load("recommendation_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")

# =====================================================
# USER INPUT
# =====================================================

print("\n" + "=" * 70)
print("CYBERSECURITY ASSESSMENT SYSTEM")
print("=" * 70)

industry = input("Industry: ").strip()
employees = int(input("Employees: "))
cloud_usage = input("Cloud Usage (Low/Medium/High): ").strip()

firewall = input("Firewall Installed (Yes/No): ").strip()
edr = input("EDR Installed (Yes/No): ").strip()
siem = input("SIEM Installed (Yes/No): ").strip()

compliance = input("Compliance Using: ").strip()
threat = input("Main Threat Concern: ").strip()
budget = input("Security Budget (Low/Medium/High): ").strip()

# =====================================================
# CREATE DATAFRAME
# =====================================================

client_data = pd.DataFrame([
    {
        "Industry": industry,
        "Employees": employees,
        "Cloud_Usage": cloud_usage,
        "Firewall_Installed": firewall,
        "EDR_Installed": edr,
        "SIEM_Installed": siem,
        "Compliance_Requirement": compliance,
        "Main_Threat_Concern": threat,
        "Security_Budget": budget
    }
])

# =====================================================
# RISK CALCULATION
# =====================================================

def calculate_risk(row):

    risk_score = 0

    if str(row["Firewall_Installed"]).lower() == "no":
        risk_score += 20

    if str(row["EDR_Installed"]).lower() == "no":
        risk_score += 30

    if str(row["SIEM_Installed"]).lower() == "no":
        risk_score += 25

    if str(row["Cloud_Usage"]).lower() == "high":
        risk_score += 10
    elif str(row["Cloud_Usage"]).lower() == "medium":
        risk_score += 5

    if str(row["Industry"]).lower() in [
        "healthcare",
        "banking",
        "government"
    ]:
        risk_score += 10

    threat_type = str(row["Main_Threat_Concern"]).lower()

    if threat_type == "ransomware":
        risk_score += 15
    elif threat_type == "data breach":
        risk_score += 10
    elif threat_type == "phishing":
        risk_score += 5
    elif threat_type == "malware":
        risk_score += 10

    compliance_req = str(row["Compliance_Requirement"]).upper()

    if compliance_req in [
        "HIPAA",
        "PCI-DSS",
        "GDPR",
        "SOC2",
        "ISO27001"
    ]:
        risk_score += 10

    if row["Employees"] > 5000:
        risk_score += 10
    elif row["Employees"] > 1000:
        risk_score += 5

    return risk_score


risk_score = calculate_risk(client_data.iloc[0])

if risk_score < 25:
    risk_level = "Low"
elif risk_score < 50:
    risk_level = "Medium"
elif risk_score < 75:
    risk_level = "High"
else:
    risk_level = "Critical"

client_data["Risk_Level"] = risk_level

# =====================================================
# RECOMMENDATION
# =====================================================

prediction = model.predict(client_data)

recommended_solution = (
    label_encoder.inverse_transform(prediction)[0]
)

# =====================================================
# PROBABILITIES
# =====================================================

probabilities = model.predict_proba(client_data)[0]

confidence = np.max(probabilities) * 100

# =====================================================
# TOP 3
# =====================================================

top3_indices = np.argsort(probabilities)[-3:][::-1]

# =====================================================
# DEBUGGING INFO
# =====================================================

print("\n")
print("=" * 70)
print("MODEL ANALYSIS")
print("=" * 70)

print(
    "Total Recommendation Classes:",
    len(label_encoder.classes_)
)

print("\nTOP 10 MODEL PROBABILITIES")
print("-" * 70)

top10_indices = np.argsort(probabilities)[-10:][::-1]

for idx in top10_indices:

    solution = label_encoder.inverse_transform([idx])[0]

    prob = probabilities[idx] * 100

    print(f"{solution} --> {prob:.2f}%")

# =====================================================
# CONFIDENCE TEXT
# =====================================================

if confidence >= 80:
    confidence_text = "Very High"
elif confidence >= 60:
    confidence_text = "High"
elif confidence >= 40:
    confidence_text = "Medium"
elif confidence >= 20:
    confidence_text = "Low"
else:
    confidence_text = "Very Low"

# =====================================================
# OUTPUT
# =====================================================

print("\n")
print("=" * 70)
print("CYBERSECURITY ASSESSMENT REPORT")
print("=" * 70)

print("\nCLIENT DETAILS")
print("-" * 70)

for col in client_data.columns:

    if col != "Risk_Level":

        print(
            f"{col}: {client_data.iloc[0][col]}"
        )

print("\nRISK ANALYSIS")
print("-" * 70)

print(f"Risk Score : {risk_score}")
print(f"Risk Level : {risk_level}")

print("\nPRIMARY RECOMMENDATION")
print("-" * 70)

print(recommended_solution)

print(
    f"\nConfidence Score : "
    f"{confidence:.2f}% ({confidence_text})"
)

print("\nTOP 3 RECOMMENDATIONS")
print("-" * 70)

for rank, idx in enumerate(top3_indices, start=1):

    solution = (
        label_encoder.inverse_transform([idx])[0]
    )

    probability = probabilities[idx] * 100

    print(
        f"{rank}. {solution} "
        f"({probability:.2f}%)"
    )

# =====================================================
# VENDOR MAPPING
# =====================================================

vendor_map = {
    "EDR": "CrowdStrike Falcon",
    "SIEM": "Microsoft Sentinel",
    "Firewall": "Palo Alto NGFW",
    "XDR": "Microsoft Defender XDR",
    "SOC": "Managed SOC Service",
    "ZTNA": "Zscaler ZTNA",
    "DLP": "Forcepoint DLP",
    "Email Security": "Proofpoint",
    "Vulnerability Management": "Tenable Nessus"
}

print("\nRECOMMENDED PRODUCTS")
print("-" * 70)

shown = set()

for idx in top3_indices:

    solution = (
        label_encoder.inverse_transform([idx])[0]
    )

    parts = solution.split("|")

    for item in parts:

        item = item.strip()

        if item in vendor_map and item not in shown:

            print(
                f"✓ {item} --> {vendor_map[item]}"
            )

            shown.add(item)

print("\n" + "=" * 70)
print("ASSESSMENT COMPLETED")
print("=" * 70)


CYBERSECURITY ASSESSMENT SYSTEM


Industry:  banking
Employees:  124
Cloud Usage (Low/Medium/High):  low
Firewall Installed (Yes/No):  no
EDR Installed (Yes/No):  no
SIEM Installed (Yes/No):  no
Compliance Using:  pcidss
Main Threat Concern:  malware
Security Budget (Low/Medium/High):  high




MODEL ANALYSIS
Total Recommendation Classes: 20

TOP 10 MODEL PROBABILITIES
----------------------------------------------------------------------
Email Security Gateway --> 5.69%
MDR Service --> 5.65%
Data Encryption Platform --> 5.59%
ZTNA Solution --> 5.50%
Threat Intelligence Platform --> 5.44%
Attack Surface Management --> 5.26%
PAM --> 5.26%
Cloud Security Platform --> 5.25%
EDR Platform --> 5.17%
XDR Platform --> 5.13%


CYBERSECURITY ASSESSMENT REPORT

CLIENT DETAILS
----------------------------------------------------------------------
Industry: banking
Employees: 124
Cloud_Usage: low
Firewall_Installed: no
EDR_Installed: no
SIEM_Installed: no
Compliance_Requirement: pcidss
Main_Threat_Concern: malware
Security_Budget: high

RISK ANALYSIS
----------------------------------------------------------------------
Risk Score : 95
Risk Level : Critical

PRIMARY RECOMMENDATION
----------------------------------------------------------------------
Email Security Gateway

Confidence S